# Python for Data: Solutions Notebook

## AJB Training Programme | 19-21 April 2026

This notebook contains the complete worked solutions for Labs A through H.
It is intended for the facilitator and reviewers only. Do not share with participants before the session.

All code uses the synthetic training datasets in the `data/` folder.

In [ ]:
# Setup: imports and paths (run this cell first)
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
OUTPUT_DIR_D1 = Path("../outputs/day1")
OUTPUT_DIR_D2 = Path("../outputs/day2")
OUTPUT_DIR_D3 = Path("../outputs/day3_pack")

OUTPUT_DIR_D1.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_D2.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_D3.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# Day 1 Solutions

---

## Lab A: Transaction Extract Triage

**Objective:** Determine whether the transactions extract is fit for first-pass analysis.

**Required output:** A triage summary table, one risk statement, and a recommendation on whether analysis should proceed.

**Stretch output:** A `rejects.csv` with reason codes.

In [ ]:
# Lab A Solution

# Step 1: Load with explicit dtypes
txns = pd.read_csv(
    DATA_DIR / "transactions.csv",
    dtype={"account_id": "string", "branch_id": "string"},
)
txns["txn_ts"] = pd.to_datetime(txns["txn_ts"], errors="coerce")
txns["amount_sar"] = pd.to_numeric(txns["amount_sar"], errors="coerce")
txns["fee_sar"] = pd.to_numeric(txns["fee_sar"], errors="coerce")

# Step 2: Triage summary
triage = pd.DataFrame([{
    "rows": len(txns),
    "columns": txns.shape[1],
    "distinct_txn_id": txns["txn_id"].nunique(),
    "null_cells": int(txns.isna().sum().sum()),
    "min_date": txns["txn_ts"].min(),
    "max_date": txns["txn_ts"].max(),
}])
print("=== Triage Summary ===")
print(triage.to_string(index=False))

# Step 3: Null rates per column
null_rates = txns.isna().mean().sort_values(ascending=False).rename("null_rate")
print("\n=== Null Rates ===")
print(null_rates[null_rates > 0].to_string())

# Step 4: Duplicate check
n_dupes = txns["txn_id"].duplicated().sum()
print(f"\nDuplicate txn_id rows: {n_dupes}")

# Step 5: Plausibility check on amounts
print(f"\nAmount range: {txns['amount_sar'].min():.2f} to {txns['amount_sar'].max():.2f}")
print(f"Negative amounts: {(txns['amount_sar'] < 0).sum()}")

# Risk statement
print("\n=== Risk Statement ===")
if n_dupes > 0:
    print(f"RISK: {n_dupes} duplicate transaction IDs found. These could inflate counts and fee totals.")
    print("RECOMMENDATION: Proceed with caution. Deduplicate before aggregation.")
else:
    print("No duplicate IDs found. File appears fit for first-pass analysis.")

# Stretch: Export rejects
dupes = txns[txns.duplicated("txn_id", keep=False)].copy()
dupes["reason_code"] = "duplicate_txn_id"

null_key_rows = txns[txns["txn_id"].isna() | txns["account_id"].isna()].copy()
null_key_rows["reason_code"] = "missing_key_field"

rejects = pd.concat([dupes, null_key_rows], ignore_index=True)
rejects.to_csv(OUTPUT_DIR_D1 / "rejects.csv", index=False)
print(f"\nRejects exported: {len(rejects)} rows")

triage.to_csv(OUTPUT_DIR_D1 / "triage_summary.csv", index=False)

## Lab B: Build Cleaning Functions

**Objective:** Create reusable cleaning logic for amounts and channel codes.

**Required output:** Cleaned columns plus a contract note describing assumptions and failure behaviour.

**Stretch output:** QC counts for unmapped values.

In [ ]:
# Lab B Solution

def clean_amount(text):
    """Convert text amount to float. Returns NaN for blank, missing, or non-numeric input."""
    if pd.isna(text):
        return np.nan
    cleaned = str(text).replace(",", "").strip()
    if cleaned == "":
        return np.nan
    try:
        return float(cleaned)
    except ValueError:
        return np.nan


CHANNEL_MAP = {
    "ATM": "ATM",
    "MOB": "Mobile",
    "BRANCH": "Branch",
    "ONLINE": "Online",
}


def clean_channel(code):
    """Map channel code to standard label. Returns 'UNMAPPED' for unrecognised values, NaN for missing."""
    if pd.isna(code):
        return np.nan
    normalised = str(code).strip().upper()
    return CHANNEL_MAP.get(normalised, "UNMAPPED")


# Apply to data
txns["amount_clean"] = txns["amount_sar"].apply(clean_amount)
txns["channel_clean"] = txns["channel"].apply(clean_channel)

# Verify
print("=== Before / After Sample ===")
print(txns[["txn_id", "amount_sar", "amount_clean", "channel", "channel_clean"]].head(10).to_string(index=False))

# QC: unmapped channels
unmapped = txns[txns["channel_clean"] == "UNMAPPED"]
print(f"\nUnmapped channel values: {len(unmapped)}")
if len(unmapped) > 0:
    print(unmapped["channel"].value_counts().to_string())

# Contract note
print("\n=== Function Contract ===")
print("clean_amount:")
print("  Input: any value from the amount_sar column (text, number, blank, or NaN)")
print("  Output: float if parseable, NaN otherwise")
print("  Assumption: commas are thousands separators, not decimal marks")
print("")
print("clean_channel:")
print("  Input: any value from the channel column")
print("  Output: standard label from CHANNEL_MAP, 'UNMAPPED' for unrecognised, NaN for missing")
print("  Assumption: input codes may have mixed case and leading/trailing spaces")

# Day 2 Solutions

---

## Lab C: Customer Quality Checks

**Objective:** Test whether the customer table can support downstream joins without distorting interpretation.

**Required output:** Duplicates and nulls summary, plus a ranked judgement on the highest analytical risk.

**Stretch output:** `dq_report.csv` with pass/fail logic.

In [ ]:
# Lab C Solution

customers = pd.read_csv(DATA_DIR / "customers.csv", dtype={"customer_id": "string"})
customers["onboarding_date"] = pd.to_datetime(customers["onboarding_date"], errors="coerce")
customers["region_clean"] = customers["region"].str.strip().str.title()

# Check 1: Duplicate customer IDs
dup_customers = customers[customers.duplicated("customer_id", keep=False)]
n_dup = len(dup_customers)
print(f"Duplicate customer_id rows: {n_dup}")

# Check 2: Null profile
null_profile = customers.isna().sum()
print("\n=== Null Counts ===")
print(null_profile[null_profile > 0].to_string())

# Check 3: Invalid region values
valid_regions = ["Riyadh", "Makkah", "Eastern", "Madinah", "Asir"]
invalid_regions = customers[~customers["region_clean"].isin(valid_regions)]
print(f"\nInvalid region values: {len(invalid_regions)}")
if len(invalid_regions) > 0:
    print(invalid_regions[["customer_id", "region", "region_clean"]].to_string(index=False))

# Risk ranking
print("\n=== Risk Ranking ===")
print("1. Duplicate customer_id (highest risk): would inflate customer counts in any join")
print("2. Invalid region values: would cause rows to drop from regional summaries")
print("3. Null fields: depends on which field; nulls in segment affect segment analysis")

# DQ report export
checks = pd.DataFrame([
    {"check": "duplicate_customer_id", "status": "fail" if n_dup > 0 else "pass", "count": n_dup},
    {"check": "invalid_region", "status": "fail" if len(invalid_regions) > 0 else "pass", "count": len(invalid_regions)},
    {"check": "null_segment", "status": "fail" if customers["segment"].isna().sum() > 0 else "pass",
     "count": int(customers["segment"].isna().sum())},
    {"check": "null_kyc_status", "status": "fail" if customers["kyc_status"].isna().sum() > 0 else "pass",
     "count": int(customers["kyc_status"].isna().sum())},
])
checks.to_csv(OUTPUT_DIR_D2 / "dq_report.csv", index=False)
print("\ndq_report.csv exported")
print(checks.to_string(index=False))

## Lab D: Branch Performance Summary

**Objective:** Create a branch KPI table that is readable and methodologically defensible.

**Required output:** `branch_kpi.csv` with txn_count, total_fee_sar, avg_ticket_sar.

**Stretch output:** active_customers, within-region rank.

In [ ]:
# Lab D Solution

accounts = pd.read_csv(DATA_DIR / "accounts.csv", dtype={"account_id": "string", "customer_id": "string"})
transactions = pd.read_csv(DATA_DIR / "transactions.csv", dtype={"account_id": "string", "branch_id": "string"})
branches = pd.read_csv(DATA_DIR / "branches.csv", dtype={"branch_id": "string"})

transactions["txn_ts"] = pd.to_datetime(transactions["txn_ts"], errors="coerce")
transactions["amount_sar"] = pd.to_numeric(transactions["amount_sar"], errors="coerce")
transactions["fee_sar"] = pd.to_numeric(transactions["fee_sar"], errors="coerce")

# Build analysis table
analysis = (
    transactions
    .merge(accounts[["account_id", "customer_id", "product_family"]], on="account_id", how="left")
    .merge(customers[["customer_id", "region_clean"]], on="customer_id", how="left")
    .merge(branches[["branch_id", "region", "city"]], on="branch_id", how="left")
)

# Filter to posted only
posted = analysis[analysis["status"] == "POSTED"].copy()

# KPI table
branch_kpi = (
    posted
    .groupby(["region", "branch_id"], dropna=False)
    .agg(
        txn_count=("txn_id", "count"),
        total_fee_sar=("fee_sar", "sum"),
        avg_ticket_sar=("amount_sar", "mean"),
        active_customers=("customer_id", "nunique"),
    )
    .reset_index()
)

# Within-region ranking
branch_kpi["rank_in_region"] = (
    branch_kpi.groupby("region")["total_fee_sar"]
    .rank(ascending=False, method="min")
    .astype(int)
)
branch_kpi = branch_kpi.sort_values(["region", "rank_in_region"])

branch_kpi.to_csv(OUTPUT_DIR_D2 / "branch_kpi.csv", index=False)
print("=== Branch KPI Table ===")
print(branch_kpi.to_string(index=False))

# Metric definitions
print("\n=== Metric Definitions ===")
print("txn_count:      Count of txn_id where status == 'POSTED', grouped by branch")
print("total_fee_sar:  Sum of fee_sar where status == 'POSTED', grouped by branch")
print("avg_ticket_sar: Mean of amount_sar where status == 'POSTED', grouped by branch")
print("active_customers: Count of distinct customer_id per branch")

## Lab E: Monthly Trend and GroupBy

**Objective:** Build a monthly aggregation that could support trend commentary.

**Required output:** Monthly totals by region, with denominator logic stated explicitly.

In [ ]:
# Lab E Solution

posted["month"] = posted["txn_ts"].dt.to_period("M").astype(str)

# Monthly aggregation by region
monthly_region = (
    posted
    .groupby(["month", "region"])
    .agg(
        txn_count=("txn_id", "count"),
        total_fee_sar=("fee_sar", "sum"),
        total_amount_sar=("amount_sar", "sum"),
        active_customers=("customer_id", "nunique"),
    )
    .reset_index()
    .sort_values(["region", "month"])
)

# Product uptake by region (denominator: distinct customers per region)
product_uptake = (
    posted
    .groupby(["region", "product_family"])
    .agg(customer_count=("customer_id", "nunique"))
    .reset_index()
)
region_totals = posted.groupby("region")["customer_id"].nunique().rename("region_customers").reset_index()
product_uptake = product_uptake.merge(region_totals, on="region")
product_uptake["uptake_rate"] = product_uptake["customer_count"] / product_uptake["region_customers"]

print("=== Monthly Summary by Region ===")
print(monthly_region.to_string(index=False))

print("\n=== Product Uptake by Region ===")
print(product_uptake.to_string(index=False))

# Denominator defence
print("\n=== Denominator Choice ===")
print("Chosen denominator: distinct customers with at least one POSTED transaction in the region")
print("Alternative: all customers assigned to the region (including those with no transactions)")
print("Rationale: using transacting customers avoids inflating the base with dormant records,")
print("but it means uptake rates reflect share-of-wallet among active customers rather than penetration.")

# Day 3 Solutions

---

## Lab F: Leadership Visualisation

**Objective:** Produce two charts suitable for a leadership pack, each supporting a specific analytical claim.

**Required output:** Charts saved as images, with titles and axis labels.

In [ ]:
# Lab F Solution

monthly_totals = (
    posted
    .groupby("month")
    .agg(txn_count=("txn_id", "count"), total_fee_sar=("fee_sar", "sum"))
    .reset_index()
)

top_branches = (
    posted
    .groupby("branch_id")
    .agg(total_fee_sar=("fee_sar", "sum"))
    .reset_index()
    .sort_values("total_fee_sar", ascending=False)
    .head(5)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Monthly transaction volume trend
axes[0].plot(monthly_totals["month"], monthly_totals["txn_count"], marker="o", color="#4fc3f7", linewidth=2)
axes[0].set_title("Monthly Transaction Volume", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Number of Transactions")
axes[0].tick_params(axis="x", rotation=45)
axes[0].grid(axis="y", alpha=0.3)

# Chart 2: Top 5 branches by fee revenue
axes[1].barh(top_branches["branch_id"], top_branches["total_fee_sar"], color="#81c784")
axes[1].set_title("Top 5 Branches by Fee Revenue (SAR)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Total Fee (SAR)")
axes[1].invert_yaxis()
axes[1].grid(axis="x", alpha=0.3)

fig.tight_layout()
fig.savefig(OUTPUT_DIR_D3 / "leadership_charts.png", dpi=150, bbox_inches="tight")
print("Charts saved to outputs/day3_pack/leadership_charts.png")

# Claim supported by each chart
print("\n=== Analytical Claims ===")
print("Chart 1: Transaction volume is [stable / growing / declining] across the period.")
print("Chart 2: The top 5 branches contribute a disproportionate share of total fee revenue.")

## Lab G: Exception Detection

**Objective:** Build a transparent, auditable exception log using a rule-based approach.

**Required output:** `exceptions.csv` with threshold, reason, and the flagged records.

In [ ]:
# Lab G Solution

branch_monthly = (
    posted
    .groupby(["branch_id", "month"])
    .agg(txn_count=("txn_id", "count"), total_fee_sar=("fee_sar", "sum"))
    .reset_index()
    .sort_values(["branch_id", "month"])
)

# Fee change month-over-month
branch_monthly["fee_change"] = branch_monthly.groupby("branch_id")["total_fee_sar"].diff()
branch_monthly["fee_pct_change"] = branch_monthly.groupby("branch_id")["total_fee_sar"].pct_change()

# Exception rules
THRESHOLD_ABS = 2.5   # SAR absolute change
THRESHOLD_PCT = 0.50  # 50% relative change

exceptions = []

# Rule 1: Large absolute fee swing
abs_exc = branch_monthly[branch_monthly["fee_change"].fillna(0).abs() > THRESHOLD_ABS].copy()
abs_exc["reason_code"] = "abs_fee_change_gt_2.5_SAR"
abs_exc["threshold"] = THRESHOLD_ABS
exceptions.append(abs_exc)

# Rule 2: Large percentage swing
pct_exc = branch_monthly[branch_monthly["fee_pct_change"].fillna(0).abs() > THRESHOLD_PCT].copy()
pct_exc["reason_code"] = "pct_fee_change_gt_50pct"
pct_exc["threshold"] = THRESHOLD_PCT
exceptions.append(pct_exc)

all_exceptions = pd.concat(exceptions, ignore_index=True).drop_duplicates(subset=["branch_id", "month", "reason_code"])
all_exceptions.to_csv(OUTPUT_DIR_D3 / "exceptions.csv", index=False)

print(f"=== Exceptions Flagged: {len(all_exceptions)} ===")
print(all_exceptions.to_string(index=False))

print("\n=== Exception Transparency ===")
print(f"Rule 1: absolute fee change > {THRESHOLD_ABS} SAR between consecutive months")
print(f"Rule 2: percentage fee change > {THRESHOLD_PCT * 100:.0f}% between consecutive months")
print("Note: thresholds are set for training purposes. In production, calibrate against historical variance.")

## Lab H: Feature Engineering and ML Handoff

**Objective:** Create a feature table and assumptions file that could be handed off to an ML team.

**Required output:** `features.csv`, `assumptions.csv`, and a two-sentence management summary.

In [ ]:
# Lab H Solution

CUT_OFF = pd.Timestamp("2026-06-30")
window = posted[posted["txn_ts"] <= CUT_OFF].copy()
window["is_digital"] = window["channel"].fillna("").str.upper().isin(["MOB", "ONLINE"])

features = (
    window
    .groupby("customer_id")
    .agg(
        txn_count=("txn_id", "count"),
        avg_ticket_sar=("amount_sar", "mean"),
        total_fee_sar=("fee_sar", "sum"),
        digital_share=("is_digital", "mean"),
        product_count=("product_family", "nunique"),
        region=("region", "first"),
        segment=("segment", "first"),
        last_txn_date=("txn_ts", "max"),
    )
    .reset_index()
)
features["cut_off_date"] = CUT_OFF.date().isoformat()
features["days_since_last_txn"] = (CUT_OFF - features["last_txn_date"]).dt.days

features.to_csv(OUTPUT_DIR_D3 / "features.csv", index=False)
print("=== Feature Table ===")
print(features.to_string(index=False))

# Assumptions log
assumptions = pd.DataFrame([
    {"feature": "cut_off_date", "assumption": "All features use data available on or before 2026-06-30"},
    {"feature": "digital_share", "assumption": "Channels MOB and ONLINE classified as digital"},
    {"feature": "txn_count", "assumption": "Only POSTED transactions counted; reversed/pending excluded"},
    {"feature": "days_since_last_txn", "assumption": "Recency calculated relative to cut-off, not run date"},
    {"feature": "product_count", "assumption": "Distinct product_family values per customer (not account count)"},
])
assumptions.to_csv(OUTPUT_DIR_D3 / "assumptions.csv", index=False)
print("\n=== Assumptions ===")
print(assumptions.to_string(index=False))

# Modelling use cases
print("\n=== Proposed Modelling Uses ===")
print("txn_count / avg_ticket_sar: activity-level features for churn or engagement scoring")
print("digital_share: channel-preference signal for digital adoption models")
print("days_since_last_txn: recency proxy for dormancy detection")
print("product_count: cross-sell breadth indicator")

# Management summary
print("\n=== Management Summary ===")
print("We have prepared a customer-level feature table covering transaction volume, channel")
print("preference, and product breadth for all customers active before 30 June 2026.")
print("Caveat: features are based on synthetic training data and should be recalculated")
print("on production extracts before any model development begins.")

---

## File Manifest

| File | Description |
|------|-------------|
| `data/transactions.csv` | Synthetic transaction extract (35 rows) |
| `data/customers.csv` | Synthetic customer records (13 rows) |
| `data/accounts.csv` | Synthetic account records (14 rows) |
| `data/branches.csv` | Synthetic branch lookup (8 rows) |
| `data/service_tickets.csv` | Synthetic support tickets (8 rows) |
| `notebooks/day1_python_foundations.ipynb` | Day 1 participant notebook (starter + prompts) |
| `notebooks/day2_numpy_pandas_core.ipynb` | Day 2 participant notebook (starter + prompts) |
| `notebooks/day3_reporting_and_handoff.ipynb` | Day 3 participant notebook (starter + prompts) |
| `notebooks/solutions.ipynb` | This file. Complete worked solutions (Labs A-H) |

---

*End of solutions notebook.*